# Problema de la Mochila 0/1 — knapPI_1_50_1000_1
### Benchmark de Pisinger | Solver: Gurobi
**Referencia:** Pisinger, D. (2005). *Where are the hard knapsack problems?* Computers & Operations Research, 32(9), 2271–2284.

**Modelo matemático:**

$$\text{Maximizar} \quad z = \sum_{i=1}^{50} p_i \cdot x_i$$

$$\text{Sujeto a} \quad \sum_{i=1}^{50} w_i \cdot x_i \leq c = 995, \qquad x_i \in \{0, 1\}$$

## 1. Instalación de librerías

In [ ]:
# gurobipy: interfaz oficial de Python para el solver Gurobi
# pandas: para leer y manipular los datos del archivo .txt como tabla
!pip install gurobipy pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 60.2 MB/s eta 0:00:00


## 2. Importación de librerías

In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB  # GRB contiene constantes: GRB.BINARY, GRB.MAXIMIZE, GRB.OPTIMAL

##2.1 Importación del Archivo

Para optimizar el ejercicio, se optó por subir el archivo `Knapsack_50items.txt` al repositorio del proyecto en Github. Se validó esta opción ya que al tener que subir el archivo manualmente, relentizaba la mejora del ejercicio.

In [ ]:
from os import path
from urllib.request import urlretrieve

DATA_FILE = "Knapsack_50items.txt"
URL = "https://raw.githubusercontent.com/CamiloLM/proyecto-optimizacion/refs/heads/main/knapsack/Knapsack_50items.txt"

if not path.isfile(DATA_FILE):
    print("Descargando archivo...")
    urlretrieve(URL, DATA_FILE)

print("Archivo listo:", DATA_FILE)

Descargando archivo...
Archivo listo: Knapsack_50items.txt


## 3. Lectura del archivo de datos

El archivo sigue el formato estándar del benchmark de Pisinger:
- Líneas 1–4: encabezados (nombre, n, capacidad, z)
- Línea 5: tiempo de cómputo (se omite)
- Líneas 6+: datos de ítems en formato `id,beneficio,peso,valor_optimo`

In [ ]:
# Se leen los encabezados línea por línea dentro de un solo bloque 'with'
# para no abrir el archivo dos veces (más limpio y eficiente)
with open(DATA_FILE) as f:
    nombre    = f.readline().strip()           # Ej: "knapPI_1_50_1000_1"
    n         = int(f.readline().split()[1])   # Número de ítems: n=50
    capacidad = int(f.readline().split()[1])   # Capacidad de la mochila: c=995
    z_opt     = int(f.readline().split()[1])   # Óptimo conocido de Pisinger: z*=8373
    f.readline()                               # Línea de tiempo — se descarta

print(f"Instancia  : {nombre}")
print(f"Ítems (n)  : {n}")
print(f"Capacidad  : {capacidad}")
print(f"Óptimo z*  : {z_opt}")

Instancia  : knapPI_1_50_1000_1
Ítems (n)  : 50
Capacidad  : 995
Óptimo z*  : 8373


In [ ]:
# (el formato Pisinger es: id,beneficio,peso,solucion_optima)
datos = pd.read_csv(
    DATA_FILE,
    sep=",",           # sep="," porque el archivo usa comas como separador entre columnas
    skiprows=5,        # skiprows=5 omite las 5 líneas de encabezado ya leídas arriba
    header=None,       # header=None porque los datos no tienen fila de títulos
    names=["id", "beneficio", "peso", "sol_pisinger"]
    # sol_pisinger: 1 si Pisinger dice que ese ítem entra en el óptimo, 0 si no
)

print(f"Ítems cargados: {len(datos)}")
datos.reset_index(drop=True).style.hide() #para eliminar el index que pandas pone por efecto y dejar solo el item que viene en el file

Ítems cargados: 50


id,beneficio,peso,sol_pisinger
1,94,485,0
2,506,326,0
3,416,248,0
4,992,421,0
5,649,322,0
6,237,795,0
7,457,43,1
8,815,845,0
9,446,955,0
10,422,252,0


## 5. Construcción del modelo en Gurobi

Se construye el modelo de Programación Lineal Entera Binaria (PLEB):

| Elemento | Definición |
|---|---|
| Variable | $x_i \in \{0,1\}$ — 1 si el ítem $i$ es seleccionado |
| F. Objetivo | $\max \sum p_i x_i$ — maximizar beneficio total |
| Restricción | $\sum w_i x_i \leq 995$ — peso total ≤ capacidad |

In [ ]:
# Se crea el modelo y se le asigna el nombre de la instancia
m = gp.Model(nombre)


# (se activa solo al llamar m.optimize())
m.Params.OutputFlag = 0     # OutputFlag=0 silencia el log de Gurobi durante la construcción

# ── VARIABLES DE DECISIÓN ──────────────────────────────────────────────
# addVars(n): crea n variables indexadas de 0 a n-1
# vtype=GRB.BINARY: restringe cada variable a {0, 1} (no se puede fraccionar un ítem)
# name="x": prefijo del nombre de cada variable (x[0], x[1], ..., x[49])
x = m.addVars(n, vtype=GRB.BINARY, name="x")

# ── FUNCIÓN OBJETIVO ───────────────────────────────────────────────────
m.setObjective(
    gp.quicksum(                              # quicksum es más eficiente que sum() nativo de Python para modelos grandes
        datos.loc[i, "beneficio"] * x[i]      # datos.loc[i, "beneficio"]: beneficio p_i del ítem i
        for i in range(n)
    ),
    GRB.MAXIMIZE                              # GRB.MAXIMIZE: indica que queremos el máximo (vs GRB.MINIMIZE)
)

# ── RESTRICCIÓN DE CAPACIDAD ───────────────────────────────────────────
# name="capacidad": nombrar la restricción permite identificarla
# análisis de sensibilidad (dual values, rangos) que genera Gurobi
m.addConstr(
    gp.quicksum(
        datos.loc[i, "peso"] * x[i]           # datos.loc[i, "peso"]: peso w_i del ítem i
        for i in range(n)
    ) <= capacidad,
    name="capacidad"   # ← siempre nombrar las restricciones
)

m.update()

print("Modelo construido correctamente.")
print(f"  Variables  : {m.NumVars}")
print(f"  Restricciones: {m.NumConstrs}")

Restricted license - for non-production use only - expires 2027-11-29
Modelo construido correctamente.
  Variables  : 50
  Restricciones: 1


## 6. Resolución del modelo

In [ ]:
# Para este tamaño (50 variables binarias) el tiempo es < 1 segundo
m.Params.OutputFlag = 1   # activar log para ver el proceso de resolución
m.optimize()              # m.optimize() ejecuta el solver Branch & Bound de Gurobi

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1 rows, 50 columns and 50 nonzeros (Max)
Model fingerprint: 0x103bfc60
Model has 50 linear objective coefficients
Variable types: 0 continuous, 50 integer (50 binary)
Coefficient statistics:
  Matrix range     [9e+00, 1e+03]
  Objective range  [7e+00, 1e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+03]

Found heuristic solution: objective 2515.0000000
Presolve removed 1 rows and 50 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 2: 8373 2515 

Optimal solution found (tolerance 1.00e-04)
Best objective 8.373000000000e+03,

## 7. Verificación del estado de la solución

In [ ]:
# IMPORTANTE: siempre verificar el status antes de leer resultados
# Si Gurobi no encontró óptimo (ej: tiempo límite, infactibilidad),
# acceder a m.ObjVal o x[i].X lanza una excepción
if m.Status == GRB.OPTIMAL:
    print("✓ Solución óptima encontrada")
elif m.Status == GRB.INFEASIBLE:
    print("✗ Modelo infactible — revisar restricciones")
elif m.Status == GRB.TIME_LIMIT:
    print("⚠ Tiempo límite alcanzado — solución puede no ser óptima")
else:
    print(f"Estado Gurobi: {m.Status} — revisar documentación")

✓ Solución óptima encontrada


## 8. Resultados y verificación contra Pisinger

In [ ]:
# Gurobi puede devolver 0.9999999 en lugar de exactamente 1.0
# El umbral 0.5 garantiza la clasificación correcta en todos los casos
seleccionados = [i for i in range(n) if x[i].X > 0.5]

# Calcular totales en un solo recorrido (eficiencia)
peso_total      = sum(datos.loc[i, "peso"]      for i in seleccionados)
beneficio_total = sum(datos.loc[i, "beneficio"] for i in seleccionados)

print("=" * 52)
print("  RESULTADOS DEL MODELO")
print("=" * 52)
print(f"  Beneficio óptimo  (Gurobi)  : {beneficio_total}")
print(f"  Beneficio óptimo  (Pisinger): {z_opt}")
print(f"  ¿Coinciden?                 : {'✓ SÍ' if beneficio_total == z_opt else '✗ NO'}")
print(f"  Peso utilizado              : {peso_total} / {capacidad}")
print(f"  Ocupación mochila           : {peso_total/capacidad*100:.1f}%")
print(f"  Ítems seleccionados         : {len(seleccionados)} de {n}")
print("=" * 52)

  RESULTADOS DEL MODELO
  Beneficio óptimo  (Gurobi)  : 8373
  Beneficio óptimo  (Pisinger): 8373
  ¿Coinciden?                 : ✓ SÍ
  Peso utilizado              : 971 / 995
  Ocupación mochila           : 97.6%
  Ítems seleccionados         : 11 de 50


## 9. Tabla de ítems seleccionados

In [ ]:
# Se construye un DataFrame con los ítems seleccionados para presentación
resultados = datos.loc[seleccionados].copy()
resultados["x_i"] = 1
resultados["p_i * x_i"] = resultados["beneficio"]
resultados["w_i * x_i"] = resultados["peso"]
resultados["ratio p/w"] = (resultados["beneficio"] / resultados["peso"]).round(2)

# Renombrar para presentación clara
resultados = resultados[["id","beneficio","peso","ratio p/w","x_i","p_i * x_i","w_i * x_i"]]
resultados.columns = ["Ítem (i)","Beneficio (pᵢ)","Peso (wᵢ)","Ratio pᵢ/wᵢ","xᵢ","pᵢ·xᵢ","wᵢ·xᵢ"]
resultados = resultados.reset_index(drop=True)

# Fila de totales
totales = pd.DataFrame([["TOTAL",beneficio_total,peso_total,"",len(seleccionados),beneficio_total,peso_total]],
                        columns=resultados.columns)
display(pd.concat([resultados, totales], ignore_index=True))

,Ítem (i),Beneficio (pᵢ),Peso (wᵢ),Ratio pᵢ/wᵢ,xᵢ,pᵢ·xᵢ,wᵢ·xᵢ
0,7,457,43,10.63,1,457,43
1,11,791,9,87.89,1,791,9
2,13,667,122,5.47,1,667,122
3,14,598,94,6.36,1,598,94
4,24,700,72,9.72,1,700,72
5,26,874,138,6.33,1,874,138
6,31,997,199,5.01,1,997,199
7,33,908,97,9.36,1,908,97
8,38,931,70,13.3,1,931,70
9,39,726,98,7.41,1,726,98


## 10. Comparación ítem a ítem con la solución de Pisinger

In [ ]:
# sol_pisinger: columna del archivo original con la solución óptima conocida
# x_solver: solución encontrada por Gurobi
# Si coinciden en todos los ítems, el modelo está formulado correctamente
datos["x_solver"] = [1 if x[i].X > 0.5 else 0 for i in range(n)]
datos["coincide"]  = datos["sol_pisinger"] == datos["x_solver"]

n_coinciden = datos["coincide"].sum()
print(f"Ítems donde Gurobi y Pisinger coinciden: {n_coinciden} / {n}")

if n_coinciden == n:
    print("✓ Solución idéntica a la de Pisinger - modelo verificado")
else:
    print("⚠ Diferencias encontradas (puede haber soluciones óptimas alternativas)")
    display(datos[~datos["coincide"]])

Ítems donde Gurobi y Pisinger coinciden: 50 / 50
✓ Solución idéntica a la de Pisinger - modelo verificado
